# Mini Quiz Generator — LangChain Chain Composition

Generates a beginner-level question and a detailed answer for any topic using two chained LLM calls (Google Gemini), composed with LangChain's LCEL runnable-pipe syntax (the current replacement for the retired `LLMChain` / `SequentialChain` classes).

**STEP 1: Set Up Environment**

In [ ]:
print("=" * 70)
print(" INSTALLING PACKAGES")
print("=" * 70)
print("This may take a minute. Please wait...")

%pip install -q langchain langchain-community langchain-google-genai python-dotenv

print("\nInstallation complete!")
print("=" * 70)

In [ ]:
# LangChain - Google Gemini chat model
from langchain_google_genai import ChatGoogleGenerativeAI

# LangChain - prompt template, output parser, and runnable composition
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Standard library - environment variable access
import os

# Secret management for a local environment - loads variables from a .env file
from dotenv import load_dotenv

In [ ]:
# ============================================================================
# Load the Google Gemini API key from a local .env file and expose it as the
# GOOGLE_API_KEY environment variable that langchain-google-genai expects.
#
# To set this up:
# 1. Get a free API key at https://aistudio.google.com/app/api-keys
# 2. Create a file named ".env" in this notebook's directory
# 3. Add one line to it:  GOOGLE_API_KEY=your-key-here
# 4. Never commit the .env file or hardcode the key in this notebook
# ============================================================================

load_dotenv()

if not os.environ.get("GOOGLE_API_KEY"):
    raise RuntimeError(
        "GOOGLE_API_KEY not found. Add it to a .env file in this directory "
        "(GOOGLE_API_KEY=your-key-here) or export it as an environment variable."
    )

print("API key configured successfully")
print("   (Key was loaded from the environment and is not displayed)")

**STEP 2: Initialize Language Model**

In [ ]:
# Initialize the Gemini model that powers both chains.
# temperature=0.7 balances accuracy (needed for correct educational content)
# with enough creativity to keep questions varied and engaging.

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.7
)

print("Gemini LLM initialized successfully")
print("   Model: gemini-3.6-flash")
print("   Temperature: 0.7 (balanced creativity)")

**STEP 3: Design Prompt Templates**

In [ ]:
# Chain 1 prompt - Question generation
# Takes a topic and produces a single beginner-level question.
# The instructions are explicit about difficulty level and output format so
# the model does not wander into intermediate/advanced territory or add
# extra commentary around the question.

question_prompt = PromptTemplate(
    input_variables=["topic"],
    template=(
        "You are an educational content creator. Generate exactly one "
        "beginner-level quiz question about the following topic: {topic}\n\n"
        "Requirements:\n"
        "- The question must be appropriate for someone new to the topic.\n"
        "- Output only the question text, with no answer, numbering, or "
        "extra commentary."
    )
)

print("Chain 1 (Question Generation) prompt template created")

In [ ]:
# Chain 2 prompt - Answer generation
# Takes the question produced by Chain 1 and produces a clear answer plus
# a short explanation, so the final quiz item is fully self-contained.

answer_prompt = PromptTemplate(
    input_variables=["question"],
    template=(
        "You are an educational content creator. Answer the following "
        "beginner-level quiz question: {question}\n\n"
        "Requirements:\n"
        "- Give a clear, direct answer first.\n"
        "- Follow it with a short explanation (2-3 sentences) that helps a "
        "beginner understand why the answer is correct."
    )
)

print("Chain 2 (Answer Generation) prompt template created")

**STEP 4: Build Individual Chains**

In [ ]:
# Chain 1: Question Generation Chain
# `prompt | llm | StrOutputParser()` is LCEL's chain-composition syntax -
# the successor to LLMChain.
question_chain = question_prompt | llm | StrOutputParser()

# Chain 2: Answer Generation Chain
answer_chain = answer_prompt | llm | StrOutputParser()

print("Individual chains created successfully")
print("   Chain 1: topic -> question")
print("   Chain 2: question -> answer")

**STEP 5: Compose Sequential Chain**

In [ ]:
# Connect both chains into a single pipeline. Each RunnablePassthrough.assign
# step keeps every existing key in the running dict and adds one more, so
# "question" (Chain 1's output) is already present in the dict Chain 2
# receives - the same automatic data routing SequentialChain used to do.

chain = (
    RunnablePassthrough.assign(question=question_chain)
    | RunnablePassthrough.assign(answer=answer_chain)
)

print("Sequential chain created successfully")
print("   Pipeline: topic -> [Chain 1] -> question -> [Chain 2] -> answer")

**STEP 6: Run Quiz Generator**

In [ ]:
# Collect a topic from the user and run the full pipeline.
# .invoke() takes a dict whose keys match the chain's input_variables
# (just "topic" here) and returns a dict containing every output_variable.

topic = input("Enter a topic: ")

response = chain.invoke({"topic": topic})

print("\n" + "=" * 70)
print(" GENERATED QUIZ")
print("=" * 70)

print("\nQuestion:")
print(response["question"])

print("\nAnswer & Explanation:")
print(response["answer"])

print("\n" + "=" * 70)

In [ ]:
# Try a few different topics to see how the generator adapts.
for sample_topic in ["Photosynthesis", "Python lists", "The French Revolution"]:
    result = chain.invoke({"topic": sample_topic})
    print(f"\nTopic: {sample_topic}")
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print("-" * 70)